In [ ]:
%pip install -U yfinance matplotlib pandas

In [ ]:
import os

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

assert isinstance(openai_endpoint, str), "ENDPOINT is missing from .env"
assert isinstance(model_deployment, str), "MODEL_DEPLOYMENT is missing from .env"

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

model_client = OpenAIChatCompletionClient(
    model=model_deployment,
    base_url=openai_endpoint,
    api_key=token_provider(),
)

print("Model client created successfully")
print("Endpoint:", openai_endpoint)
print("Model:", model_deployment)

In [ ]:
%pip install asyncio-atexit docker

In [4]:
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

In [11]:
from pathlib import Path

from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor


# Create the working directory.
coding_directory = Path("coding")
coding_directory.mkdir(parents=True, exist_ok=True)


# Create the local code executor.
executor = LocalCommandLineCodeExecutor(
    timeout=120,
    work_dir=coding_directory,
)


# Agent responsible for writing and correcting the code.
code_writer_agent = AssistantAgent(
    name="code_writer_agent",
    model_client=model_client,
    system_message=(
        "You are a Python financial-analysis coding assistant. "
        "Write complete executable Python code inside one markdown "
        "python code block. "
        "Use yfinance to retrieve stock-market data. "
        "Use pandas for calculations and matplotlib for plotting. "
        "Save every requested chart in the current working directory. "
        "Do not use plt.show(). "
        "After the executor confirms that the code completed successfully "
        "and the requested file exists, reply with TERMINATE. "
        "If execution fails, carefully inspect the error and provide a "
        "complete corrected Python code block."
    ),
)


# Agent responsible for executing markdown code blocks.
code_executor_agent = CodeExecutorAgent(
    name="code_executor_agent",
    code_executor=executor,
)


# Stop when the writer returns TERMINATE.
termination_condition = TextMentionTermination("TERMINATE")


# Writer creates code, executor runs it, writer reviews the result.
team = RoundRobinGroupChat(
    participants=[
        code_writer_agent,
        code_executor_agent,
    ],
    termination_condition=termination_condition,
    max_turns=8,
)

print("Writer and executor agents created successfully")

Writer and executor agents created successfully


C:\Users\a855099\AppData\Local\Temp\ipykernel_14136\549628910.py:16: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(
C:\Users\a855099\AppData\Local\Temp\ipykernel_14136\549628910.py:16: UserWarning: The current event loop policy is not WindowsProactorEventLoopPolicy. This may cause issues with subprocesses. Try setting the event loop policy to WindowsProactorEventLoopPolicy. For example: `asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())`. See https://docs.python.org/3/library/asyncio-eventloop.html#asyncio.ProactorEventLoop.
  executor = LocalCommandLineCodeExecutor(
C:\Users\a855099\AppData\Local\Temp\ipykernel_14136\549628910.py:43: UserWarning: No approval function set for CodeExecutorAgent. This means code will be execute

In [6]:
import datetime

today = datetime.date.today()
start_of_year = datetime.date(today.year, 1, 1)

task = f"""
Today is {today}.

Create a line chart comparing year-to-date percentage gains for:

- NVIDIA: NVDA
- Tesla: TSLA

Requirements:

1. Download daily historical data from {start_of_year} through {today}
   using yfinance.
2. Use adjusted closing prices when available. If the returned data does
   not contain an adjusted-close field, use the close field.
3. Remove rows where both ticker values are missing.
4. Normalize each ticker to percentage return using:

   ((current price / first valid price) - 1) * 100

5. Plot both percentage-return series on one line chart.
6. Add a horizontal zero-percent reference line.
7. Include a title, axis labels, legend, and grid.
8. Save the image as ytd_stock_gains.png in the current working directory.
9. Print the first available price, latest available price, latest date,
   and calculated YTD percentage gain for each ticker.
10. Do not call plt.show().
11. Put all required Python code in one markdown Python code block.
12. When execution succeeds and the file is confirmed, reply TERMINATE.
"""

result = await Console(
    team.run_stream(task=task)
)

---------- TextMessage (user) ----------

Today is 2026-09-26.

Create a line chart comparing year-to-date percentage gains for:

- NVIDIA: NVDA
- Tesla: TSLA

Requirements:

1. Download daily historical data from 2026-01-01 through 2026-09-26
   using yfinance.
2. Use adjusted closing prices when available. If the returned data does
   not contain an adjusted-close field, use the close field.
3. Remove rows where both ticker values are missing.
4. Normalize each ticker to percentage return using:

   ((current price / first valid price) - 1) * 100

5. Plot both percentage-return series on one line chart.
6. Add a horizontal zero-percent reference line.
7. Include a title, axis labels, legend, and grid.
8. Save the image as ytd_stock_gains.png in the current working directory.
9. Print the first available price, latest available price, latest date,
   and calculated YTD percentage gain for each ticker.
10. Do not call plt.show().
11. Put all required Python code in one markdown Python 

In [10]:
from pathlib import Path
from IPython.display import Image, display

chart_path = Path("coding") / "ytd_stock_gains.png"

if chart_path.exists():
    print("Chart created successfully:")
    print(chart_path.resolve())
    display(Image(filename=str(chart_path)))
else:
    print("The chart was not created.")
    print("Check the writer/executor output above for an execution error.")

The chart was not created.
Check the writer/executor output above for an execution error.


In [9]:
import datetime
from pathlib import Path
from typing_extensions import Annotated

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from autogen_agentchat.agents import AssistantAgent


def create_ytd_stock_chart(
    symbols: Annotated[
        list[str],
        "Stock ticker symbols, for example ['NVDA', 'TSLA']",
    ],
    filename: Annotated[
        str,
        "Output PNG filename",
    ],
) -> str:
    """Download stock data and create a YTD percentage-return chart."""

    if not symbols:
        return "No stock symbols were provided."

    today = datetime.date.today()
    start_date = datetime.date(today.year, 1, 1)

    data = yf.download(
        symbols,
        start=str(start_date),
        end=str(today + datetime.timedelta(days=1)),
        auto_adjust=False,
        progress=False,
    )

    if data.empty:
        return "No stock data was returned."

    if isinstance(data.columns, pd.MultiIndex):
        if "Adj Close" in data.columns.get_level_values(0):
            prices = data["Adj Close"]
        elif "Close" in data.columns.get_level_values(0):
            prices = data["Close"]
        else:
            return "Neither Adj Close nor Close was found."
    else:
        price_column = "Adj Close" if "Adj Close" in data.columns else "Close"
        prices = data[[price_column]].copy()
        prices.columns = symbols[:1]

    if isinstance(prices, pd.Series):
        prices = prices.to_frame(name=symbols[0])

    prices = prices.dropna(how="all")

    if prices.empty:
        return "No usable prices were returned."

    ytd_returns = pd.DataFrame(index=prices.index)
    summary_lines = []

    for symbol in symbols:
        if symbol not in prices.columns:
            summary_lines.append(f"{symbol}: no data returned")
            continue

        series = prices[symbol].dropna()

        if series.empty:
            summary_lines.append(f"{symbol}: no valid prices returned")
            continue

        first_price = float(series.iloc[0])
        latest_price = float(series.iloc[-1])

        ytd_returns[symbol] = (
            (series / first_price) - 1
        ) * 100

        ytd_gain = ((latest_price / first_price) - 1) * 100

        summary_lines.append(
            f"{symbol}: first={first_price:.2f}, "
            f"latest={latest_price:.2f}, "
            f"latest date={series.index[-1].date()}, "
            f"YTD gain={ytd_gain:.2f}%"
        )

    if ytd_returns.empty:
        return "\n".join(summary_lines)

    output_path = Path("coding") / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(11, 6))

    for symbol in ytd_returns.columns:
        plt.plot(
            ytd_returns.index,
            ytd_returns[symbol],
            label=symbol,
            linewidth=2,
        )

    plt.axhline(
        y=0,
        color="black",
        linewidth=1,
        linestyle="--",
    )

    plt.title("Year-to-Date Stock Performance")
    plt.xlabel("Date")
    plt.ylabel("Gain (%)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

    return (
        f"Chart saved successfully to {output_path}.\n"
        + "\n".join(summary_lines)
    )


financial_agent = AssistantAgent(
    name="financial_agent",
    model_client=model_client,
    tools=[create_ytd_stock_chart],
    reflect_on_tool_use=True,
    max_tool_iterations=3,
    system_message=(
        "You are a financial data assistant. "
        "Use create_ytd_stock_chart whenever the user requests a "
        "year-to-date stock comparison. "
        "Do not invent market values. "
        "Report only the values returned by the tool. "
        "This is descriptive analysis, not investment advice."
    ),
)


financial_result = await financial_agent.run(
    task=(
        "Create a YTD percentage-gain chart for NVDA and TSLA. "
        "Save it as stock_prices_YTD_plot.png."
    )
)

print(financial_result.messages[-1].content)

Matplotlib is building the font cache; this may take a moment.


I created the YTD percentage-return chart and saved it as stock_prices_YTD_plot.png (path: coding\stock_prices_YTD_plot.png).

Values used (from the generated data):
- NVDA: first=188.41, latest=225.07 (2026-09-25), YTD gain = 19.46%
- TSLA: first=438.07, latest=372.11 (2026-09-25), YTD gain = -15.06%

Let me know if you want the chart in a different format, a file upload, or a brief interpretation of the results.
